<div align="center">

# 🚕 Yandex Go Analytics

## 6 · Data Enrichment

*Join every trip with its driver's profile in a single, analysis-ready table.*

![layer](https://img.shields.io/badge/layer-gold-FFB300)
![operation](https://img.shields.io/badge/operation-LEFT%20JOIN-00ADD4)
![table](https://img.shields.io/badge/table-enriched_trips-2E7D32)

</div>

| ◀ Previous | 🏠 Home | Next ▶ |
|:--|:--:|--:|
| [5 · Aggregations & Optimizations](./5.%20Aggregations%20%26%20Optimizations.ipynb) | [Project Overview](./0.%20Project%20Overview.ipynb) | [7 · Refactor via Delta Live Tables](./7.%20Refactor%20via%20Delta%20Live%20Tables.ipynb) |

**Progress** &nbsp; ▰▰▰▰▰▰▱▱  **6 / 8**

---

## 🎯 Task requirements

- [x] Generate a **synthetic `driver_id`** for the taxi trips
- [x] Perform a **`LEFT JOIN`** between the trips and the driver profiles
- [x] Save the unified, enriched dataset as **`gold.enriched_trips`**

## 🧩 How it is solved

**Synthetic key.** The trip generator assigns every trip a `driver_id` drawn from a *deterministic* UUID pool
(fixed Faker seed). The driver generator uses the **same** pool, so trips and driver profiles share a join key
without any real-world identifiers.

```text
 silver.taxi  (stream, de-duplicated by trip id)
      │  driver_id
      ▼
   LEFT JOIN  ◀──────────  silver.drivers  (static snapshot, one row per driver id)
      │
      ▼
 gold.enriched_trips   =   all trip columns  +  driver_name · driver_car_number
                                                driver_experience · driver_rating [· driver_phone]
```

| Design choice | Why |
|:--|:--|
| `LEFT JOIN` | A trip must survive even when its driver profile is missing or was rejected by Silver |
| Stream ⨝ static | New trips flow in incrementally; the (small) driver dimension is read as a snapshot |
| `driver_` prefix | Keeps trip and driver columns unambiguous |
| `driver_phone` only if present | The `phone` column was rolled back in notebook 3, so it is added only if it exists in Silver |

## 1️⃣ Connect to Spark

In [1]:
from pyspark.sql import functions as F

from src.core.config import get_catalog, get_environment
from src.core.infra import DatabricksConnectionService

conn = DatabricksConnectionService(app_name="yandex-go-enrichment")
conn.connect()
spark = conn.spark

CATALOG = get_catalog()
print(f"✅ Connected | Spark {spark.version} | catalog = {CATALOG} | env = {get_environment()}")

2026-10-09 17:26:53.406 | INFO     | src.core.infra.connection.databricks_connector:connect:56 - SparkSession ready | app=yandex-go-enrichment | version=4.2.0 | runtime=serverless-connect


✅ Connected | Spark 4.2.0 | catalog = yandex_go_prod | env = prod


## 2️⃣ Build `gold.enriched_trips`

In [2]:
from src.core.etl.gold import EnrichedTripsGoldService

EnrichedTripsGoldService().run(spark)

enriched = spark.table(f"{CATALOG}.gold.enriched_trips")
print(f"gold.enriched_trips: {enriched.count():,} rows | {len(enriched.columns)} columns")
display(enriched.toPandas())

2026-10-09 17:26:53.419 | INFO     | src.core.etl.gold.trip_driver:run:34 - Reading stream from silver.taxi via checkpoints...
2026-10-09 17:27:23.691 | INFO     | src.core.etl.gold.trip_driver:run:73 - Enriched yandex_go_trips_pipeline successfully written!


gold.enriched_trips: 3,789 rows | 32 columns


,driver_id,id,user_id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,...,_rescued_data,_ingested_at,_source_file,payment_type_description,_processed_at,driver_name,driver_car_number,driver_experience,driver_rating,_calculated_at
0,4068cacc-1f7d-446f-9716-8ab3a4351379,4464a3eb-eebe-4082-ad16-25569445cc0c,bc0a6a5d-6e99-4e3e-a3b1-37fc0a3450fc,2,2026-10-01 00:01:53,2026-10-01 00:11:53,4,15.66,3,N,...,None,2026-10-09 14:16:49.570,/Volumes/yandex_go_prod/raw_files/landing/parq...,Cash,2026-10-09 14:21:49.814,None,None,NaN,NaN,2026-10-09 14:26:57.141
1,ef4b9bf0-7a09-4ac9-a4b7-f979a8e45869,849cf6c0-9a12-4958-aa01-df7f540bae43,1a432f0a-7daa-49f0-80b6-fce2de53790a,2,2026-10-02 09:13:56,2026-10-02 09:28:56,4,3.77,5,N,...,None,2026-10-09 14:16:49.570,/Volumes/yandex_go_prod/raw_files/landing/parq...,Other,2026-10-09 14:21:49.814,None,None,NaN,NaN,2026-10-09 14:26:57.141
2,b6c4fd42-81d2-48e1-adb0-3041c6637d6a,ae77b807-2d00-4c62-90ff-1360e9519b35,7ed70ed7-b194-490b-a961-929e546e035a,2,2026-10-06 17:31:07,2026-10-06 18:08:07,1,8.80,2,N,...,None,2026-10-09 14:16:49.570,/Volumes/yandex_go_prod/raw_files/landing/parq...,Credit Card,2026-10-09 14:21:49.814,Shane Garcia,K765XE,6.0,4.70,2026-10-09 14:26:57.141
3,06ae4148-efcc-46b6-89a9-c1e44df180d0,ce5f2e12-1e16-4eb5-963b-5cff56bc3e66,e8dd5e5a-1712-4b16-a1a4-344fbb7bee03,1,2026-10-08 09:46:08,2026-10-08 09:54:08,6,26.66,1,N,...,None,2026-10-09 14:16:49.570,/Volumes/yandex_go_prod/raw_files/landing/parq...,Other,2026-10-09 14:21:49.814,Brian Chen,Y767CE,11.0,4.63,2026-10-09 14:26:57.141
4,f3a3c571-7476-4899-b5a3-adb3254a9493,4abc924c-8c26-4f0e-9479-73fa6290164d,fc377a4c-4a15-444d-85e7-ce8a3a578a8e,2,2026-10-09 02:17:22,2026-10-09 02:29:22,1,10.52,1,N,...,None,2026-10-09 14:16:49.570,/Volumes/yandex_go_prod/raw_files/landing/parq...,Credit Card,2026-10-09 14:21:49.814,None,None,NaN,NaN,2026-10-09 14:26:57.141
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3784,9395925b-d7aa-4fce-99df-f86829364cbb,0ea234e5-4090-4ea9-94d2-2d27aa7f7153,3a3c8a71-ff57-4e2b-8991-ab9bebc2026f,2,2026-10-09 13:43:30,2026-10-09 14:15:30,6,28.53,1,Y,...,None,2026-10-09 14:16:49.570,/Volumes/yandex_go_prod/raw_files/landing/parq...,Credit Card,2026-10-09 14:21:49.814,Cheryl Fitzgerald,A662MO,19.0,4.45,2026-10-09 14:26:57.141
3785,e6d47d34-7e54-44d7-8d15-76338379e94e,6f751632-a628-418f-be84-1ee3a5c4e6ef,96fc734d-a003-4d28-8a8f-3653c9af18f8,1,2026-10-08 20:06:06,2026-10-08 20:29:06,2,19.18,5,Y,...,None,2026-10-09 14:16:49.570,/Volumes/yandex_go_prod/raw_files/landing/parq...,Credit Card,2026-10-09 14:21:49.814,Michael Jones,P264TT,11.0,4.65,2026-10-09 14:26:57.141
3786,8dca0621-ccfd-437f-9893-63c95a3b2c8a,671da6eb-997e-4fc4-add1-99e8ba07af1c,a6f2f7b8-0cf3-4b58-9910-8be58ce21ea3,2,2026-10-02 04:09:25,2026-10-02 04:18:25,1,4.53,1,Y,...,None,2026-10-09 14:16:49.570,/Volumes/yandex_go_prod/raw_files/landing/parq...,Other,2026-10-09 14:21:49.814,Tammy Gonzales,Y472OB,17.0,4.94,2026-10-09 14:26:57.141
3787,6e149b07-7559-4c0e-8dbb-1cb464be22a3,6f43b7ea-b9c7-4d26-b112-032954054e24,f1043785-658b-4523-a014-1de9f54ad0a2,1,2026-10-09 00:50:46,2026-10-09 01:06:46,3,16.70,3,N,...,None,2026-10-09 14:16:49.570,/Volumes/yandex_go_prod/raw_files/landing/parq...,Credit Card,2026-10-09 14:21:49.814,Anthony Woods,O559OC,6.0,4.82,2026-10-09 14:26:57.141


## 3️⃣ Verify the join

In [3]:
def check(label: str, condition: bool) -> None:
    print(("✅" if condition else "❌"), label)
    assert condition, label

silver_trip_count = spark.table(f"{CATALOG}.silver.taxi").select("id").distinct().count()
enriched_count = enriched.count()

check(
    f"row-preserving: {enriched_count:,} enriched rows == {silver_trip_count:,} distinct Silver trips",
    enriched_count == silver_trip_count,
)
check("one row per trip id", enriched_count == enriched.select("id").distinct().count())
check(
    "driver columns are present",
    {"driver_name", "driver_car_number", "driver_experience", "driver_rating"} <= set(enriched.columns),
)

✅ row-preserving: 3,789 enriched rows == 3,789 distinct Silver trips
✅ one row per trip id
✅ driver columns are present


### 🎯 Match rate

Trips whose driver has no Silver profile keep `NULL` driver columns — the signature of a `LEFT JOIN`.

In [4]:
match = (
    enriched.agg(
        F.count("*").alias("total_trips"),
        F.sum(F.when(F.col("driver_name").isNotNull(), 1).otherwise(0)).alias("with_driver"),
        F.sum(F.when(F.col("driver_name").isNull(), 1).otherwise(0)).alias("without_driver"),
    )
    .withColumn("match_rate_pct", F.round(F.col("with_driver") / F.col("total_trips") * 100, 1))
)
display(match.toPandas())

assert match.first()["with_driver"] > 0, "No trip was matched with a driver profile"
print("✅ Trips are enriched with driver profiles")

,total_trips,with_driver,without_driver,match_rate_pct
0,3789,2399,1390,63.3


✅ Trips are enriched with driver profiles


### 🏆 What the enriched table unlocks

Revenue per driver — impossible without the join.

In [5]:
top_drivers = (
    enriched.filter(F.col("driver_name").isNotNull())
    .groupBy("driver_id", "driver_name", "driver_experience", "driver_rating")
    .agg(
        F.count("*").alias("trips"),
        F.round(F.sum("fare_amount"), 2).alias("revenue"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
    )
    .orderBy(F.col("revenue").desc())
)
display(top_drivers.toPandas())

,driver_id,driver_name,driver_experience,driver_rating,trips,revenue,avg_distance
0,fd35bbb7-fb13-4e8b-84dc-57de990e75ea,Meredith Richardson,18,4.38,52,2856.55,16.43
1,d639834f-9110-45ef-8e16-5fb01870b05a,Matthew Wall,21,4.88,46,2756.28,17.50
2,90e25b2d-051d-4c86-aa8e-0a7c71085730,Amanda Henderson,21,4.85,47,2713.65,17.06
3,6576923c-81e1-4733-91dd-85fdb4164b76,Garrett Rodriguez,12,4.89,43,2624.14,17.23
4,2e3ee8c4-1fc4-43ca-a371-db42a606e864,Dorothy Martinez,11,4.34,46,2575.50,16.76
...,...,...,...,...,...,...,...
58,b064fbfd-1b31-48ca-bb0a-2baa27810664,Lisa Mendez,15,4.31,35,1425.12,11.81
59,0c119c08-391c-4954-9dba-78d2943c10ee,Christine Wallace,24,4.77,31,1422.99,13.64
60,6d23c048-8a60-49c1-896f-6a3c66eae3d9,Crystal Robinson,3,4.98,29,1288.36,12.49
61,9e6bfff5-b7a1-4d67-bf41-58639bedfe39,Garrett Morrison,13,4.42,25,1225.11,14.41


---

## ✅ Checkpoint

- [x] Synthetic `driver_id` shared by trips and driver profiles
- [x] `LEFT JOIN` of `silver.taxi` with `silver.drivers`
- [x] `gold.enriched_trips` written; no trip lost, match rate reported

> **Next up ▶ [7 · Refactor via Delta Live Tables](./7.%20Refactor%20via%20Delta%20Live%20Tables.ipynb)** — recreate the whole pipeline declaratively with Delta Live Tables.

<div align="center"><sub>Yandex Go Analytics · notebook 6 of 8 · <a href="https://github.com/stanislavpanfilenko/YandexGoAnalytics">GitHub</a>